# Giai đoạn 1: Chuyển đổi BPI 2017 sang định dạng OCEL 2.0

**Vấn đề với Traditional Process Mining:**
- BPI 2017 có 3 luồng đối tượng song song: `Application (A_)`, `Offer (O_)`, `Workflow (W_)`
- Traditional PM buộc phải chọn 1 Case ID duy nhất → mất thông tin quan hệ 1-Nhiều
- 1 Application có thể sinh ra **nhiều Offers** → bị phẳng hóa (flattening)

**Giải pháp: Object-Centric Event Log (OCEL 2.0)**
- Mỗi sự kiện có thể liên kết với **nhiều đối tượng thuộc nhiều loại**
- Giữ nguyên quan hệ `Application ↔ Offer` mà không mất thông tin

```
Application_001  ──►  Offer_A  (Returned → Accepted)
                 ──►  Offer_B  (Refused)
                 ──►  Offer_C  (Accepted ✓)
```

In [ ]:
import pandas as pd
import numpy as np
import pm4py
import json
import os
import warnings
warnings.filterwarnings('ignore')

print(f"pm4py version: {pm4py.__version__}")
print("✅ Môi trường sẵn sàng")

## Bước 1: Đọc và khám phá dữ liệu

In [ ]:
# Đọc dữ liệu gốc
print("⏳ Đang tải dữ liệu...")
df = pd.read_csv('../data/bpi-challenge-2017/bpi_2017_cleaned.csv')
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True, errors='coerce')
df = df.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

print(f"✅ Tổng số sự kiện: {len(df):,}")
print(f"✅ Tổng số Applications: {df['case:concept:name'].nunique():,}")
print(f"\n📋 Các cột hiện có:")
print(df.columns.tolist())

In [ ]:
# Phân loại sự kiện theo đối tượng
df_app      = df[df['concept:name'].str.startswith('A_')].copy()
df_offer    = df[df['concept:name'].str.startswith('O_')].copy()
df_workflow = df[df['concept:name'].str.startswith('W_')].copy()

print("📊 Phân bố sự kiện theo loại đối tượng:")
print(f"  • Application events (A_): {len(df_app):,}")
print(f"  • Offer events (O_):       {len(df_offer):,}")
print(f"  • Workflow events (W_):    {len(df_workflow):,}")
print()

# Thống kê OfferID
offers_with_id = df_offer[df_offer['OfferID'].notna()]
print(f"📦 Tổng số Offers duy nhất: {offers_with_id['OfferID'].nunique():,}")
print(f"📦 Offers trên mỗi Application (trung bình): {offers_with_id.groupby('case:concept:name')['OfferID'].nunique().mean():.2f}")

In [ ]:
# Phân phối số Offer trên mỗi Application
import matplotlib.pyplot as plt

offers_per_app = offers_with_id.groupby('case:concept:name')['OfferID'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(offers_per_app.values, bins=range(1, offers_per_app.max()+2),
             color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Phân phối số Offer trên mỗi Application', fontsize=13)
axes[0].set_xlabel('Số Offer')
axes[0].set_ylabel('Số Application')
axes[0].grid(axis='y', alpha=0.3)

# Value counts
vc = offers_per_app.value_counts().sort_index()
axes[1].bar(vc.index, vc.values, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Số Application theo số Offer (Tỷ lệ %)', fontsize=13)
axes[1].set_xlabel('Số Offer')
axes[1].set_ylabel('Số Application')
for i, (x, y) in enumerate(zip(vc.index, vc.values)):
    pct = y / len(offers_per_app) * 100
    axes[1].text(x, y + 20, f'{pct:.1f}%', ha='center', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('offers_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Đã lưu biểu đồ: offers_distribution.png")

## Bước 2: Xây dựng bảng đối tượng (Object Tables)

In [ ]:
# ── Bảng đối tượng: Application ─────────────────────────────────────────────
app_objects = df.groupby('case:concept:name').agg(
    LoanGoal       = ('case:LoanGoal', 'first'),
    ApplicationType= ('case:ApplicationType', 'first'),
    RequestedAmount= ('case:RequestedAmount', 'first')
).reset_index()
app_objects.rename(columns={'case:concept:name': 'object_id'}, inplace=True)
app_objects['object_type'] = 'application'

print(f"✅ Application objects: {len(app_objects):,}")
display(app_objects.head(3))

In [ ]:
# ── Bảng đối tượng: Offer ────────────────────────────────────────────────────
offer_events = df_offer[df_offer['OfferID'].notna()].copy()

offer_objects = offer_events.groupby('OfferID').agg(
    OfferedAmount      = ('OfferedAmount', 'first'),
    NumberOfTerms      = ('NumberOfTerms', 'first'),
    MonthlyCost        = ('MonthlyCost', 'first'),
    CreditScore        = ('CreditScore', 'first'),
    Accepted           = ('Accepted', 'first'),
    Selected           = ('Selected', 'first'),
    parent_application = ('case:concept:name', 'first')   # quan hệ Application → Offer
).reset_index()
offer_objects.rename(columns={'OfferID': 'object_id'}, inplace=True)
offer_objects['object_type'] = 'offer'

print(f"✅ Offer objects: {len(offer_objects):,}")
display(offer_objects.head(3))

## Bước 3: Xây dựng bảng sự kiện (Event Table)

In [ ]:
# Mỗi sự kiện cần có:
#   event_id, activity, timestamp, resource
#   + danh sách đối tượng liên kết (omap)

df_events = df.copy()
df_events['event_id'] = df_events['EventID'].astype(str)

# Xây dựng cột omap: danh sách ID đối tượng liên quan đến sự kiện
def build_omap(row):
    """Xác định các đối tượng liên quan đến 1 sự kiện."""
    objects = []
    # Mọi sự kiện đều liên kết với Application
    objects.append({'objectId': row['case:concept:name'], 'objectType': 'application'})
    # Sự kiện Offer liên kết thêm với Offer cụ thể
    if pd.notna(row['OfferID']):
        objects.append({'objectId': row['OfferID'], 'objectType': 'offer'})
    return objects

print("⏳ Đang xây dựng bảng sự kiện OCEL...")
df_events['omap'] = df_events.apply(build_omap, axis=1)

print(f"✅ Tổng số sự kiện OCEL: {len(df_events):,}")
display(df_events[['event_id', 'concept:name', 'time:timestamp', 'omap']].head(5))

## Bước 4: Tạo OCEL DataFrame và xuất file

In [ ]:
# Chuyển sang định dạng pm4py OCEL
# pm4py 2.7+ hỗ trợ tạo OCEL từ DataFrame

from pm4py.objects.ocel.obj import OCEL
import pm4py.objects.ocel.util.ocel_consistency as consistency

# Tạo events DataFrame
events_df = pd.DataFrame({
    'ocel:eid':       df_events['event_id'],
    'ocel:activity':  df_events['concept:name'],
    'ocel:timestamp': df_events['time:timestamp'],
    'ocel:omap':      df_events['omap'],
    # Thuộc tính sự kiện bổ sung
    'resource':       df_events['org:resource'],
    'lifecycle':      df_events['lifecycle:transition'],
})

# Tạo objects DataFrame
app_obj_df = pd.DataFrame({
    'ocel:oid':  app_objects['object_id'],
    'ocel:type': app_objects['object_type'],
    'LoanGoal':        app_objects['LoanGoal'],
    'ApplicationType': app_objects['ApplicationType'],
    'RequestedAmount': app_objects['RequestedAmount'],
})

offer_obj_df = pd.DataFrame({
    'ocel:oid':  offer_objects['object_id'],
    'ocel:type': offer_objects['object_type'],
    'OfferedAmount': offer_objects['OfferedAmount'],
    'NumberOfTerms': offer_objects['NumberOfTerms'],
    'MonthlyCost':   offer_objects['MonthlyCost'],
    'Accepted':      offer_objects['Accepted'],
})

objects_df = pd.concat([app_obj_df, offer_obj_df], ignore_index=True)

print(f"✅ Events DataFrame: {len(events_df):,} rows")
print(f"✅ Objects DataFrame: {len(objects_df):,} rows")
display(objects_df.head(3))

In [ ]:
# Tạo OCEL object bằng pm4py
ocel = pm4py.read.read_ocel2_xml   # placeholder — xem bước export bên dưới

# pm4py 2.7 API: tạo OCEL từ DataFrame
ocel = pm4py.convert.convert_log_to_ocel(
    events_df,
    object_types=['application', 'offer'],
    ev_activity_key='ocel:activity',
    ev_timestamp_key='ocel:timestamp',
    ev_id_key='ocel:eid',
)

print("✅ OCEL object tạo thành công!")
print(ocel)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Phương pháp thay thế (robust hơn): Tạo OCEL thủ công theo chuẩn pm4py 2.7
# ═══════════════════════════════════════════════════════════════════════════════

# Tạo relations DataFrame (bảng quan hệ Event ↔ Object)
relations_records = []
for _, row in df_events.iterrows():
    for obj in row['omap']:
        relations_records.append({
            'ocel:eid':      row['event_id'],
            'ocel:activity': row['concept:name'],
            'ocel:timestamp':row['time:timestamp'],
            'ocel:oid':      obj['objectId'],
            'ocel:type':     obj['objectType'],
        })

relations_df = pd.DataFrame(relations_records)
print(f"✅ Relations DataFrame: {len(relations_df):,} rows")
display(relations_df.head(5))

In [ ]:
# Tạo OCEL object theo API chuẩn của pm4py 2.7
from pm4py.objects.ocel.obj import OCEL

ocel = OCEL(
    events=events_df,
    objects=objects_df,
    relations=relations_df,
)

print("✅ OCEL object khởi tạo thành công!")
print(f"  • Số sự kiện: {len(ocel.events):,}")
print(f"  • Số đối tượng: {len(ocel.objects):,}")
print(f"  • Số quan hệ E-O: {len(ocel.relations):,}")

## Bước 5: Lưu OCEL ra file

In [ ]:
os.makedirs('output', exist_ok=True)

# Xuất dạng JSON (dễ đọc, inspect)
output_json = 'output/bpi2017_ocel.jsonocel'
pm4py.write.write_ocel_json(ocel, output_json)
print(f"✅ Đã xuất OCEL JSON: {output_json}")

# Xuất dạng XML (chuẩn OCEL 2.0 chính thức)
output_xml = 'output/bpi2017_ocel2.xml'
pm4py.write.write_ocel2_xml(ocel, output_xml)
print(f"✅ Đã xuất OCEL 2.0 XML: {output_xml}")

# Lưu relations dưới dạng CSV để dùng trong các notebook tiếp theo
relations_df.to_csv('output/bpi2017_relations.csv', index=False)
events_df.drop(columns=['ocel:omap']).to_csv('output/bpi2017_events.csv', index=False)
objects_df.to_csv('output/bpi2017_objects.csv', index=False)
print(f"✅ Đã lưu CSV hỗ trợ vào thư mục output/")

## Bước 6: Kiểm tra nhanh tính nhất quán

In [ ]:
# Xác minh: 1 Application → Nhiều Offers
app_offer_map = offer_objects.groupby('parent_application')['object_id'].apply(list)
multi_offer_apps = app_offer_map[app_offer_map.apply(len) > 1]

print(f"✅ Applications có nhiều hơn 1 Offer: {len(multi_offer_apps):,}")
print(f"   ({len(multi_offer_apps)/len(app_objects)*100:.1f}% tổng số Applications)")

# Xem ví dụ
example_app = multi_offer_apps.index[0]
example_offers = multi_offer_apps.iloc[0]
print(f"\n📌 Ví dụ Application '{example_app}' có {len(example_offers)} Offers:")
for oid in example_offers:
    row = offer_objects[offer_objects['object_id'] == oid].iloc[0]
    print(f"   → {oid} | Amount: {row['OfferedAmount']} | Accepted: {row['Accepted']}")

In [ ]:
print("=" * 60)
print("🏁 OCEL Conversion hoàn tất!")
print("=" * 60)
print("\nFile output:")
print("  📄 output/bpi2017_ocel.jsonocel   ← OCEL JSON")
print("  📄 output/bpi2017_ocel2.xml       ← OCEL 2.0 XML")
print("  📄 output/bpi2017_relations.csv   ← Bảng quan hệ E-O")
print("  📄 output/bpi2017_events.csv      ← Bảng sự kiện")
print("  📄 output/bpi2017_objects.csv     ← Bảng đối tượng")
print("\n▶️  Chạy tiếp: ocpm_discovery.ipynb")